# Week 8: Hypothesis Testing — Applied Statistics with Python (2026)

Last week we learned how to *estimate* population parameters with confidence intervals. This week we go further: we learn to make **decisions** about those parameters using **hypothesis testing**. Is a new drug effective? Is the average delivery time different from what's advertised? Does gender affect product preference? Hypothesis testing gives us a rigorous framework to answer such questions.

| # | Topic | Key Concepts |
|---|-------|--------------|
| 1 | Hypothesis Testing Framework | H₀ vs H₁, significance level, decision rules |
| 2 | Test Statistics & P-values | How p-values work, common misinterpretations |
| 3 | Type I & Type II Errors | α, β, statistical power |
| 4 | One-Sample Z-Test | σ known, large sample |
| 5 | One-Sample T-Test | σ unknown, `scipy.stats.ttest_1samp` |
| 6 | Two-Sample T-Test | Independent groups comparison |
| 7 | Paired T-Test | Before/after, matched pairs |
| 8 | Chi-Square Test of Independence | Categorical variables |
| 9 | Chi-Square Goodness of Fit | Observed vs expected frequencies |
| 10 | Effect Size | Cohen's d, practical vs statistical significance |
| 11 | Multiple Testing Problem | Bonferroni, FDR correction |
| 12 | Practical Example | Complete hypothesis testing pipeline |
| 13 | Summary | Key functions and concepts |
| 14 | Homework | Practice exercises |

### Import all necessary libraries for this week's analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set consistent style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Reproducibility
np.random.seed(42)

print('All libraries loaded successfully!')

---
## 1. Hypothesis Testing Framework

Hypothesis testing is a formal procedure for deciding between two competing claims about a population parameter.

### The Steps:
1. **State hypotheses**: H₀ (null) and H₁ (alternative)
2. **Choose significance level** α (usually 0.05)
3. **Collect data** and compute a **test statistic**
4. **Calculate the p-value** (or compare to critical value)
5. **Make a decision**: reject H₀ or fail to reject H₀

### Null vs Alternative Hypothesis

| | Null Hypothesis (H₀) | Alternative Hypothesis (H₁) |
|--|----------------------|-----------------------------|
| Meaning | "No effect", "no difference", status quo | "There IS an effect/difference" |
| Contains | Equality (=, ≤, ≥) | Inequality (≠, <, >) |
| Burden of proof | Assumed true until evidence says otherwise | Must be supported by data |

### Types of Tests

| Test Type | H₁ | When to Use |
|-----------|----|-----------|
| Two-tailed | μ ≠ μ₀ | "Different from" (most common) |
| Left-tailed | μ < μ₀ | "Less than" |
| Right-tailed | μ > μ₀ | "Greater than" |

Visualize the rejection regions for two-tailed, left-tailed, and right-tailed hypothesis tests to understand how decision boundaries work.

In [ ]:
# Visualize rejection regions for different test types
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
x = np.linspace(-4, 4, 300)
y = stats.norm.pdf(x)
alpha = 0.05

# Two-tailed test
ax = axes[0]
ax.plot(x, y, 'k-', linewidth=2)
z_crit = stats.norm.ppf(1 - alpha/2)  # 1.96
ax.fill_between(x, y, where=(x <= -z_crit), color='red', alpha=0.4, label=f'Reject H₀ (α/2={alpha/2})')
ax.fill_between(x, y, where=(x >= z_crit), color='red', alpha=0.4)
ax.fill_between(x, y, where=(x > -z_crit) & (x < z_crit), color='steelblue', alpha=0.2, label='Fail to reject H₀')
ax.set_title(f'Two-Tailed Test\nH₁: μ ≠ μ₀', fontsize=13)
ax.axvline(-z_crit, color='red', linestyle='--', alpha=0.7)
ax.axvline(z_crit, color='red', linestyle='--', alpha=0.7)
ax.text(-z_crit, -0.02, f'-{z_crit:.2f}', ha='center', fontsize=9, color='red')
ax.text(z_crit, -0.02, f'{z_crit:.2f}', ha='center', fontsize=9, color='red')
ax.legend(fontsize=9, loc='upper right')

# Left-tailed test
ax = axes[1]
ax.plot(x, y, 'k-', linewidth=2)
z_left = stats.norm.ppf(alpha)  # -1.645
ax.fill_between(x, y, where=(x <= z_left), color='red', alpha=0.4, label=f'Reject H₀ (α={alpha})')
ax.fill_between(x, y, where=(x > z_left), color='steelblue', alpha=0.2, label='Fail to reject H₀')
ax.set_title(f'Left-Tailed Test\nH₁: μ < μ₀', fontsize=13)
ax.axvline(z_left, color='red', linestyle='--', alpha=0.7)
ax.text(z_left, -0.02, f'{z_left:.2f}', ha='center', fontsize=9, color='red')
ax.legend(fontsize=9, loc='upper right')

# Right-tailed test
ax = axes[2]
ax.plot(x, y, 'k-', linewidth=2)
z_right = stats.norm.ppf(1 - alpha)  # 1.645
ax.fill_between(x, y, where=(x >= z_right), color='red', alpha=0.4, label=f'Reject H₀ (α={alpha})')
ax.fill_between(x, y, where=(x < z_right), color='steelblue', alpha=0.2, label='Fail to reject H₀')
ax.set_title(f'Right-Tailed Test\nH₁: μ > μ₀', fontsize=13)
ax.axvline(z_right, color='red', linestyle='--', alpha=0.7)
ax.text(z_right, -0.02, f'{z_right:.2f}', ha='center', fontsize=9, color='red')
ax.legend(fontsize=9, loc='upper right')

for ax in axes:
    ax.set_xlabel('Test Statistic (z)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_ylim(-0.04, 0.42)

plt.tight_layout()
plt.show()

---
## 2. Test Statistics & P-values

### Test Statistic
A **test statistic** converts your sample data into a single number that measures how far the sample result is from what H₀ predicts:

$$\text{Test Statistic} = \frac{\text{Sample Statistic} - \text{Null Value}}{\text{Standard Error}}$$

### P-value
The **p-value** is the probability of observing a test statistic as extreme as (or more extreme than) the one computed, **assuming H₀ is true**.

| P-value | Interpretation |
|---------|---------------|
| Small (< α) | Strong evidence against H₀ → Reject H₀ |
| Large (≥ α) | Weak evidence against H₀ → Fail to reject H₀ |

### Common Misinterpretations (AVOID these!)
- p-value is NOT the probability that H₀ is true
- p-value is NOT the probability of making an error
- A small p-value does NOT mean the effect is large or important
- "Fail to reject H₀" does NOT mean H₀ is true

Demonstrate what a p-value means through simulation: under the null hypothesis, how often do we see a result as extreme as our observation?

In [ ]:
# P-value demonstration: coin flip fairness test
# H₀: p = 0.5 (fair coin)  H₁: p ≠ 0.5
# Observation: 62 heads out of 100 flips

n_flips = 100
observed_heads = 62

# Simulate 100,000 experiments under H₀ (fair coin)
np.random.seed(42)
n_sim = 100_000
simulated_heads = np.random.binomial(n_flips, 0.5, n_sim)

# P-value: proportion of simulations as extreme as observed
# Two-tailed: count both ≥62 and ≤38 (equally extreme in opposite direction)
p_value_sim = np.mean((simulated_heads >= observed_heads) | 
                       (simulated_heads <= n_flips - observed_heads))

# Exact p-value using binomial test
p_value_exact = stats.binomtest(observed_heads, n_flips, 0.5, alternative='two-sided').pvalue

fig, ax = plt.subplots(figsize=(12, 5))
counts, bins, _ = ax.hist(simulated_heads, bins=range(30, 72), color='steelblue', 
                           alpha=0.6, edgecolor='white', density=True, label='Simulated under H₀')

# Highlight extreme regions
ax.hist(simulated_heads[simulated_heads >= observed_heads], bins=range(30, 72),
        color='red', alpha=0.7, edgecolor='white', density=True, label=f'≥ {observed_heads} heads')
ax.hist(simulated_heads[simulated_heads <= n_flips - observed_heads], bins=range(30, 72),
        color='red', alpha=0.7, edgecolor='white', density=True, label=f'≤ {n_flips - observed_heads} heads')

ax.axvline(observed_heads, color='darkred', linestyle='--', linewidth=2)
ax.axvline(n_flips - observed_heads, color='darkred', linestyle='--', linewidth=2)
ax.set_xlabel('Number of Heads', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'P-value = {p_value_sim:.4f} (simulated) vs {p_value_exact:.4f} (exact)\n'
             f'How likely is 62/100 heads if the coin is fair?', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'Observed: {observed_heads} heads in {n_flips} flips')
print(f'P-value (simulation): {p_value_sim:.4f}')
print(f'P-value (exact):      {p_value_exact:.4f}')
print(f'Decision at α=0.05:   {"Reject H₀" if p_value_exact < 0.05 else "Fail to reject H₀"}')

Show how the p-value changes as the observed result becomes more extreme — building intuition for what makes evidence "strong".

In [ ]:
# How p-value changes with different observed results
observed_values = range(50, 71)
p_values = [stats.binomtest(k, 100, 0.5, alternative='two-sided').pvalue 
            for k in observed_values]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['red' if p < 0.05 else 'steelblue' for p in p_values]
ax.bar(observed_values, p_values, color=colors, alpha=0.7, edgecolor='white')
ax.axhline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
ax.set_xlabel('Observed Heads (out of 100)', fontsize=12)
ax.set_ylabel('P-value', fontsize=12)
ax.set_title('P-value vs. Observed Result (H₀: p = 0.5)', fontsize=13)
ax.legend(fontsize=12)

# Annotate the boundary
for k, p in zip(observed_values, p_values):
    if abs(p - 0.05) < 0.02:
        ax.annotate(f'{k} heads\np={p:.3f}', xy=(k, p), xytext=(k+2, p+0.1),
                   arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

plt.tight_layout()
plt.show()

---
## 3. Type I & Type II Errors

No test is perfect. There are two kinds of mistakes we can make:

| | H₀ is actually TRUE | H₀ is actually FALSE |
|--|---------------------|---------------------|
| **Reject H₀** | Type I Error (α) — False positive | Correct! (Power = 1-β) |
| **Fail to reject H₀** | Correct! | Type II Error (β) — False negative |

- **α (Type I error rate)**: Probability of rejecting H₀ when it's true. We *choose* this (usually 0.05).
- **β (Type II error rate)**: Probability of failing to reject H₀ when it's false. Depends on effect size and n.
- **Power = 1 - β**: Probability of correctly detecting a real effect.

**Trade-off:** Decreasing α increases β (and vice versa). The only way to reduce both is to increase sample size.

Simulate Type I and Type II error rates to show that they match the theoretical values.

In [ ]:
# Simulate Type I error: test with data from H₀-true population
np.random.seed(42)
n = 30
alpha = 0.05
n_sim = 10_000
true_mu = 100  # H₀: μ = 100
sigma = 15

# Type I: data comes from μ=100 (H₀ is TRUE)
type1_rejections = 0
for _ in range(n_sim):
    sample = np.random.normal(true_mu, sigma, n)  # H₀ is true!
    t_stat, p_val = stats.ttest_1samp(sample, true_mu)
    if p_val < alpha:
        type1_rejections += 1

print(f'=== Type I Error (False Positive) ===')
print(f'H₀ is TRUE (μ = {true_mu}), testing at α = {alpha}')
print(f'Rejections: {type1_rejections}/{n_sim} = {type1_rejections/n_sim:.4f}')
print(f'Expected (α): {alpha}')

# Type II: data comes from μ=105 (H₀ is FALSE, real effect)
real_mu = 105  # True mean is actually 105
type2_failures = 0
for _ in range(n_sim):
    sample = np.random.normal(real_mu, sigma, n)  # H₀ is FALSE!
    t_stat, p_val = stats.ttest_1samp(sample, true_mu)  # Still testing H₀: μ=100
    if p_val >= alpha:
        type2_failures += 1  # Failed to detect the real effect

print(f'\n=== Type II Error (False Negative) ===')
print(f'H₀ is FALSE (true μ = {real_mu}, H₀ says μ = {true_mu})')
print(f'Failed to reject: {type2_failures}/{n_sim} = {type2_failures/n_sim:.4f}')
print(f'Power (1-β): {1 - type2_failures/n_sim:.4f}')

Visualize how statistical power depends on sample size and effect size — key concepts for study design.

In [ ]:
# Power analysis: how power depends on n and effect size
from scipy.stats import norm

def compute_power(n, effect_size, alpha=0.05):
    """Compute power for a one-sample z-test (two-sided)."""
    z_crit = norm.ppf(1 - alpha/2)
    # Non-centrality parameter
    ncp = effect_size * np.sqrt(n)
    # Power = P(reject H₀ | H₁ is true)
    power = 1 - norm.cdf(z_crit - ncp) + norm.cdf(-z_crit - ncp)
    return power

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Power vs sample size for different effect sizes
n_vals = np.arange(5, 301)
for d, label, color in [(0.2, 'Small (d=0.2)', 'steelblue'), 
                         (0.5, 'Medium (d=0.5)', 'coral'),
                         (0.8, 'Large (d=0.8)', 'seagreen')]:
    powers = [compute_power(n, d) for n in n_vals]
    axes[0].plot(n_vals, powers, linewidth=2, label=label, color=color)

axes[0].axhline(0.80, color='gray', linestyle='--', alpha=0.5, label='Power = 0.80')
axes[0].set_xlabel('Sample Size (n)', fontsize=12)
axes[0].set_ylabel('Power (1 - β)', fontsize=12)
axes[0].set_title('Power vs Sample Size', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_ylim(0, 1.02)

# Right: Power vs effect size for different sample sizes
d_vals = np.linspace(0, 1.5, 200)
for n, label, color in [(20, 'n=20', 'steelblue'), 
                         (50, 'n=50', 'coral'),
                         (100, 'n=100', 'seagreen'),
                         (200, 'n=200', 'purple')]:
    powers = [compute_power(n, d) for d in d_vals]
    axes[1].plot(d_vals, powers, linewidth=2, label=label, color=color)

axes[1].axhline(0.80, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel("Effect Size (Cohen's d)", fontsize=12)
axes[1].set_ylabel('Power (1 - β)', fontsize=12)
axes[1].set_title('Power vs Effect Size', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.show()

Visualize the overlapping distributions under H₀ and H₁ to show where Type I and Type II errors occur.

In [ ]:
# Visualize H₀ and H₁ distributions with error regions
fig, ax = plt.subplots(figsize=(12, 6))

x = np.linspace(-4, 8, 500)
mu0, mu1 = 0, 3  # H₀ and H₁ means (effect size d = 3/1 = 3 for visibility)

# Distribution under H₀
y0 = stats.norm.pdf(x, mu0, 1)
ax.plot(x, y0, 'b-', linewidth=2.5, label='Distribution under H₀')

# Distribution under H₁
y1 = stats.norm.pdf(x, mu1, 1)
ax.plot(x, y1, 'r-', linewidth=2.5, label='Distribution under H₁')

# Critical value (one-tailed for clarity)
z_crit = stats.norm.ppf(0.95)  # 1.645
ax.axvline(z_crit, color='black', linestyle='--', linewidth=1.5, label=f'Critical value = {z_crit:.2f}')

# Type I error (α): area under H₀ beyond critical value
x_fill_alpha = np.linspace(z_crit, 4.5, 100)
ax.fill_between(x_fill_alpha, stats.norm.pdf(x_fill_alpha, mu0, 1), 
                color='blue', alpha=0.3, label='Type I Error (α)')

# Type II error (β): area under H₁ below critical value
x_fill_beta = np.linspace(-4, z_crit, 100)
ax.fill_between(x_fill_beta, stats.norm.pdf(x_fill_beta, mu1, 1), 
                color='red', alpha=0.3, label='Type II Error (β)')

# Power: area under H₁ beyond critical value
x_fill_power = np.linspace(z_crit, 8, 100)
ax.fill_between(x_fill_power, stats.norm.pdf(x_fill_power, mu1, 1),
                color='green', alpha=0.3, label='Power (1-β)')

ax.set_xlabel('Test Statistic', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Type I Error, Type II Error, and Power', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

---
## 4. One-Sample Z-Test

The **z-test** is used when:
- Population σ is known
- Sample size is large (n ≥ 30)

Test statistic: $z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$

In practice, the z-test is rarely used (because σ is usually unknown), but it's important for understanding the logic of hypothesis testing.

Perform a one-sample z-test: a factory claims average battery life is 500 hours. We test this claim with a sample of 40 batteries.

In [ ]:
# One-sample Z-test example
# Factory claims: battery life = 500 hours (μ₀ = 500)
# Known σ = 30 hours (from long production history)
# Our sample of 40 batteries
np.random.seed(42)
battery_sample = np.random.normal(loc=490, scale=30, size=40)  # True mean is 490

mu_0 = 500  # Null hypothesis value
sigma = 30  # Known population σ
n = len(battery_sample)
x_bar = battery_sample.mean()

# Z-test statistic
z_stat = (x_bar - mu_0) / (sigma / np.sqrt(n))

# P-values for different alternatives
p_two_tail = 2 * stats.norm.cdf(-abs(z_stat))    # H₁: μ ≠ 500
p_left_tail = stats.norm.cdf(z_stat)              # H₁: μ < 500
p_right_tail = 1 - stats.norm.cdf(z_stat)         # H₁: μ > 500

print(f'Sample mean: {x_bar:.2f} hours')
print(f'H₀: μ = {mu_0}')
print(f'Z-statistic: {z_stat:.4f}')
print(f'\nP-values:')
print(f'  Two-tailed (H₁: μ ≠ 500): {p_two_tail:.4f} → {"Reject H₀" if p_two_tail < 0.05 else "Fail to reject H₀"}')
print(f'  Left-tailed (H₁: μ < 500): {p_left_tail:.4f} → {"Reject H₀" if p_left_tail < 0.05 else "Fail to reject H₀"}')
print(f'  Right-tailed (H₁: μ > 500): {p_right_tail:.4f} → {"Reject H₀" if p_right_tail < 0.05 else "Fail to reject H₀"}')

Visualize the z-test result, showing where our test statistic falls relative to the rejection region.

In [ ]:
# Visualize the z-test
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-4, 4, 300)
y = stats.norm.pdf(x)

ax.plot(x, y, 'k-', linewidth=2)
ax.fill_between(x, y, where=(x <= -1.96), color='red', alpha=0.3, label='Rejection region')
ax.fill_between(x, y, where=(x >= 1.96), color='red', alpha=0.3)

# Mark our test statistic
ax.axvline(z_stat, color='darkblue', linewidth=2.5, linestyle='-', 
           label=f'z = {z_stat:.2f} (our result)')
ax.plot(z_stat, stats.norm.pdf(z_stat), 'bo', markersize=10, zorder=5)

# P-value shading (two-tailed)
ax.fill_between(x, y, where=(x <= z_stat), color='blue', alpha=0.15)
ax.fill_between(x, y, where=(x >= -z_stat), color='blue', alpha=0.15)

ax.set_xlabel('Z-statistic', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'One-Sample Z-Test: z = {z_stat:.2f}, p = {p_two_tail:.4f}', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. One-Sample T-Test

The **t-test** is the workhorse of hypothesis testing. Used when:
- Population σ is **unknown** (almost always)
- Data is approximately normal (or n ≥ 30 by CLT)

Test statistic: $t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}$ with df = n - 1

Python: `scipy.stats.ttest_1samp(sample, popmean)`

### 5.1 Basic One-Sample T-Test

A coffee shop claims their large coffee contains 16 oz. We measure 25 cups to test this claim.

Perform a one-sample t-test to check whether the coffee shop's claim of 16 oz is accurate.

In [ ]:
# Coffee shop test: H₀: μ = 16 oz
np.random.seed(42)
coffee_cups = np.random.normal(loc=15.6, scale=0.5, size=25)  # Slightly underfilling!

# Two-tailed t-test
t_stat, p_value = stats.ttest_1samp(coffee_cups, popmean=16)

print(f'Sample size: n = {len(coffee_cups)}')
print(f'Sample mean: {coffee_cups.mean():.3f} oz')
print(f'Sample std:  {coffee_cups.std(ddof=1):.3f} oz')
print(f'\nH₀: μ = 16 oz')
print(f'H₁: μ ≠ 16 oz')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'\nDecision at α=0.05: {"Reject H₀" if p_value < 0.05 else "Fail to reject H₀"}')
if p_value < 0.05:
    print('→ Evidence suggests the coffee shop is NOT filling cups to 16 oz!')

### 5.2 One-Tailed T-Test

`scipy.stats.ttest_1samp` always returns a two-tailed p-value. For one-tailed tests, we need to adjust.

Show how to perform one-tailed tests using scipy: the key is dividing the two-tailed p-value by 2 and checking the direction of the t-statistic.

In [ ]:
# One-tailed test: is the coffee shop UNDER-filling? (H₁: μ < 16)
t_stat, p_two_tail = stats.ttest_1samp(coffee_cups, popmean=16)

# For left-tailed (H₁: μ < μ₀):
# If t_stat < 0: p_left = p_two_tail / 2
# If t_stat > 0: p_left = 1 - p_two_tail / 2
if t_stat < 0:
    p_left = p_two_tail / 2
else:
    p_left = 1 - p_two_tail / 2

# For right-tailed (H₁: μ > μ₀):
p_right = 1 - p_left

print(f't-statistic: {t_stat:.4f}')
print(f'\nTwo-tailed p-value (H₁: μ ≠ 16):   {p_two_tail:.6f}')
print(f'Left-tailed p-value (H₁: μ < 16):   {p_left:.6f}')
print(f'Right-tailed p-value (H₁: μ > 16):  {p_right:.6f}')

# Alternative: use the 'alternative' parameter (SciPy >= 1.7)
t_stat2, p_less = stats.ttest_1samp(coffee_cups, popmean=16, alternative='less')
t_stat3, p_greater = stats.ttest_1samp(coffee_cups, popmean=16, alternative='greater')
print(f'\nUsing alternative parameter:')
print(f'  alternative="less":    p = {p_less:.6f}')
print(f'  alternative="greater": p = {p_greater:.6f}')

### 5.3 Checking Assumptions

The t-test assumes the data is approximately normally distributed (or n is large enough for CLT). Let's verify this.

Check normality of our coffee cup data using a histogram, Q-Q plot, and the Shapiro-Wilk test.

In [ ]:
# Check t-test assumptions
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. Histogram + KDE
axes[0].hist(coffee_cups, bins=10, density=True, color='steelblue', alpha=0.7, edgecolor='white')
x_range = np.linspace(coffee_cups.min() - 0.5, coffee_cups.max() + 0.5, 100)
axes[0].plot(x_range, stats.norm.pdf(x_range, coffee_cups.mean(), coffee_cups.std()), 
             'r-', linewidth=2, label='Normal fit')
axes[0].set_title('Histogram + Normal Curve', fontsize=13)
axes[0].set_xlabel('Volume (oz)', fontsize=11)
axes[0].legend(fontsize=10)

# 2. Q-Q plot
stats.probplot(coffee_cups, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot', fontsize=13)

# 3. Box plot
axes[2].boxplot(coffee_cups, vert=True)
axes[2].axhline(16, color='red', linestyle='--', label='Claimed μ = 16')
axes[2].set_title('Box Plot', fontsize=13)
axes[2].set_ylabel('Volume (oz)', fontsize=11)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

# Shapiro-Wilk test for normality
shapiro_stat, shapiro_p = stats.shapiro(coffee_cups)
print(f'Shapiro-Wilk test: W = {shapiro_stat:.4f}, p = {shapiro_p:.4f}')
print(f'Normality assumption: {"Satisfied" if shapiro_p > 0.05 else "Violated"} (p {"≥" if shapiro_p >= 0.05 else "<"} 0.05)')

---
## 6. Two-Sample T-Test (Independent Samples)

Compare means of **two independent groups**:
- H₀: μ₁ = μ₂ (no difference between groups)
- H₁: μ₁ ≠ μ₂ (groups differ)

| Variant | Assumption | Python |
|---------|-----------|--------|
| Student's t-test | Equal variances | `ttest_ind(a, b, equal_var=True)` |
| Welch's t-test | Unequal variances (default) | `ttest_ind(a, b, equal_var=False)` |

**Welch's t-test is preferred** as the default because it doesn't assume equal variances and performs well even when variances are equal.

### 6.1 Comparing Two Teaching Methods

Test whether a new teaching method produces different exam scores compared to the traditional method.

Perform a two-sample t-test comparing exam scores between traditional and new teaching methods.

In [ ]:
# Two teaching methods
np.random.seed(42)
traditional = np.random.normal(loc=72, scale=10, size=35)  # Traditional method
new_method = np.random.normal(loc=78, scale=12, size=30)    # New method

# Welch's t-test (default, doesn't assume equal variances)
t_stat, p_value = stats.ttest_ind(traditional, new_method, equal_var=False)

print('=== Two-Sample T-Test (Welch\'s) ===')
print(f'Traditional: n={len(traditional)}, mean={traditional.mean():.2f}, std={traditional.std(ddof=1):.2f}')
print(f'New Method:  n={len(new_method)}, mean={new_method.mean():.2f}, std={new_method.std(ddof=1):.2f}')
print(f'\nDifference in means: {new_method.mean() - traditional.mean():.2f}')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.4f}')
print(f'\nDecision at α=0.05: {"Reject H₀" if p_value < 0.05 else "Fail to reject H₀"}')

Visualize the distributions of both groups side by side to see the overlap and difference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping histograms
ax = axes[0]
ax.hist(traditional, bins=15, alpha=0.6, color='steelblue', label='Traditional', density=True, edgecolor='white')
ax.hist(new_method, bins=15, alpha=0.6, color='coral', label='New Method', density=True, edgecolor='white')
ax.axvline(traditional.mean(), color='steelblue', linestyle='--', linewidth=2)
ax.axvline(new_method.mean(), color='coral', linestyle='--', linewidth=2)
ax.set_xlabel('Exam Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'Score Distributions (p = {p_value:.4f})', fontsize=13)
ax.legend(fontsize=11)

# Box plots
ax = axes[1]
bp = ax.boxplot([traditional, new_method], labels=['Traditional', 'New Method'],
                patch_artist=True, widths=0.6)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('coral')
for box in bp['boxes']:
    box.set_alpha(0.6)
ax.set_ylabel('Exam Score', fontsize=12)
ax.set_title('Score Comparison by Method', fontsize=13)

plt.tight_layout()
plt.show()

### 6.2 Testing for Equal Variances: Levene's Test

Before using Student's t-test (which assumes equal variances), we can check with Levene's test.

Perform Levene's test for equality of variances, and show how the choice between Student's and Welch's t-test depends on the result.

In [ ]:
# Levene's test: H₀: σ₁² = σ₂²
levene_stat, levene_p = stats.levene(traditional, new_method)

print(f'Levene\'s test: F = {levene_stat:.4f}, p = {levene_p:.4f}')
print(f'Equal variances? {"Yes" if levene_p > 0.05 else "No"}')

# Compare Student's vs Welch's
t_student, p_student = stats.ttest_ind(traditional, new_method, equal_var=True)
t_welch, p_welch = stats.ttest_ind(traditional, new_method, equal_var=False)

print(f'\nStudent\'s t-test: t = {t_student:.4f}, p = {p_student:.4f}')
print(f'Welch\'s t-test:   t = {t_welch:.4f}, p = {p_welch:.4f}')
print(f'\n→ In practice, always use Welch\'s (equal_var=False) — it\'s safer and nearly as powerful.')

---
## 7. Paired T-Test

The **paired t-test** is used when the two samples are **not independent** — they come from the same subjects measured at two different times or under two conditions.

Examples:
- Before vs after treatment on the same patients
- Test scores before vs after a training program
- Left hand vs right hand reaction times

The paired t-test works on the **differences** d = x₁ - x₂, reducing it to a one-sample t-test on d.

Test statistic: $t = \frac{\bar{d}}{s_d / \sqrt{n}}$ with df = n - 1

### 7.1 Before/After Study

A weight loss program claims to reduce weight. We measure 20 participants before and after the 3-month program.

Perform a paired t-test on before/after weight measurements and visualize individual changes.

In [ ]:
# Weight loss program: before vs after
np.random.seed(42)
n = 20
before = np.random.normal(85, 12, n)  # Weight before (kg)
# After: most lose 2-5 kg, some don't change
weight_change = np.random.normal(-3, 3, n)  # Negative = weight loss
after = before + weight_change

# Paired t-test
t_stat, p_value = stats.ttest_rel(before, after)

# One-tailed: we expect weight LOSS (before > after), so H₁: μ_before > μ_after
# Equivalently: H₁: mean(before - after) > 0
t_stat_one, p_one = stats.ttest_rel(before, after, alternative='greater')

differences = before - after
print('=== Paired T-Test: Weight Loss Program ===')
print(f'Before: mean = {before.mean():.2f} kg')
print(f'After:  mean = {after.mean():.2f} kg')
print(f'Mean difference (before - after): {differences.mean():.2f} kg')
print(f'\nTwo-tailed: t = {t_stat:.4f}, p = {p_value:.4f}')
print(f'One-tailed (H₁: weight loss): t = {t_stat_one:.4f}, p = {p_one:.4f}')
print(f'\nDecision at α=0.05: {"Reject H₀" if p_one < 0.05 else "Fail to reject H₀"}')
if p_one < 0.05:
    print('→ Evidence supports that the program leads to weight loss!')

Visualize individual participant changes and the distribution of differences to understand the paired data structure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Before-After lines (spaghetti plot)
ax = axes[0]
for i in range(n):
    color = 'green' if after[i] < before[i] else 'red'  # Green = lost weight
    ax.plot([0, 1], [before[i], after[i]], color=color, alpha=0.5, linewidth=1.5)
    ax.plot([0, 1], [before[i], after[i]], 'o', color=color, markersize=5)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Before', 'After'], fontsize=12)
ax.set_ylabel('Weight (kg)', fontsize=12)
ax.set_title('Individual Weight Changes\n(green=loss, red=gain)', fontsize=13)

# 2. Distribution of differences
ax = axes[1]
ax.hist(differences, bins=10, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No change')
ax.axvline(differences.mean(), color='orange', linestyle='-', linewidth=2, 
           label=f'Mean diff = {differences.mean():.2f}')
ax.set_xlabel('Weight Difference (Before - After)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Differences', fontsize=13)
ax.legend(fontsize=10)

# 3. Scatter plot: before vs after
ax = axes[2]
ax.scatter(before, after, color='steelblue', s=60, alpha=0.7)
lims = [min(before.min(), after.min()) - 5, max(before.max(), after.max()) + 5]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='No change line')
ax.set_xlabel('Before (kg)', fontsize=12)
ax.set_ylabel('After (kg)', fontsize=12)
ax.set_title('Before vs After', fontsize=13)
ax.legend(fontsize=10)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### 7.2 Why Paired Matters: Paired vs Independent

Using a paired test when the data is paired gives much more statistical power because it removes between-subject variability.

Compare the results of a paired t-test vs an independent t-test on the same data to demonstrate why the correct test choice matters.

In [ ]:
# Why paired matters: compare paired vs independent test on the same data
t_paired, p_paired = stats.ttest_rel(before, after)
t_indep, p_indep = stats.ttest_ind(before, after)

print('Same data, different tests:')
print(f'  Paired t-test:      t = {t_paired:.4f}, p = {p_paired:.4f}')
print(f'  Independent t-test: t = {t_indep:.4f}, p = {p_indep:.4f}')
print(f'\n→ The paired test has a MUCH smaller p-value!')
print(f'→ This is because it removes between-subject variability.')
print(f'\nBetween-subject std (raw weights): {np.std(np.concatenate([before,after]), ddof=1):.2f}')
print(f'Within-subject std (differences):   {differences.std(ddof=1):.2f}')

---
## 8. Chi-Square Test of Independence

The **chi-square test of independence** tests whether two **categorical** variables are associated.

- H₀: The variables are independent (no association)
- H₁: The variables are associated

Test statistic: $\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$

Where O = observed frequency, E = expected frequency under independence.

Python: `scipy.stats.chi2_contingency(contingency_table)`

### 8.1 Gender and Product Preference

Test whether gender is associated with product preference (A, B, or C) in a customer survey.

Create a contingency table and perform the chi-square test of independence.

In [ ]:
# Contingency table: Gender × Product Preference
observed = pd.DataFrame(
    [[120, 90, 40],
     [80, 110, 60]],
    index=['Male', 'Female'],
    columns=['Product A', 'Product B', 'Product C']
)

print('Observed Frequencies:')
print(observed)
print(f'\nTotal: {observed.values.sum()}')

# Chi-square test
chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print(f'\nExpected Frequencies (under H₀: independence):')
print(pd.DataFrame(expected.round(1), index=observed.index, columns=observed.columns))

print(f'\n=== Chi-Square Test of Independence ===')
print(f'χ² = {chi2:.4f}')
print(f'p-value = {p_value:.4f}')
print(f'Degrees of freedom = {dof}')
print(f'\nDecision at α=0.05: {"Reject H₀" if p_value < 0.05 else "Fail to reject H₀"}')
if p_value < 0.05:
    print('→ Gender and product preference ARE associated!')

Visualize the observed vs expected frequencies and the percentage distribution by gender.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Grouped bar chart: observed vs expected
ax = axes[0]
x = np.arange(len(observed.columns))
width = 0.2

ax.bar(x - 1.5*width, observed.loc['Male'], width, color='steelblue', alpha=0.8, label='Male (observed)')
ax.bar(x - 0.5*width, expected[0], width, color='steelblue', alpha=0.3, label='Male (expected)', edgecolor='steelblue', linewidth=2)
ax.bar(x + 0.5*width, observed.loc['Female'], width, color='coral', alpha=0.8, label='Female (observed)')
ax.bar(x + 1.5*width, expected[1], width, color='coral', alpha=0.3, label='Female (expected)', edgecolor='coral', linewidth=2)

ax.set_xticks(x)
ax.set_xticklabels(observed.columns, fontsize=11)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Observed vs Expected Frequencies', fontsize=13)
ax.legend(fontsize=9)

# 2. Heatmap of standardized residuals
ax = axes[1]
std_residuals = (observed.values - expected) / np.sqrt(expected)
std_res_df = pd.DataFrame(std_residuals.round(2), index=observed.index, columns=observed.columns)
sns.heatmap(std_res_df, annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', ax=ax, cbar_kws={'label': 'Std. Residual'})
ax.set_title('Standardized Residuals\n(shows WHERE the association is)', fontsize=13)

plt.tight_layout()
plt.show()

print('Standardized residuals > |2| indicate cells contributing most to the association.')

### 8.2 Effect Size for Chi-Square: Cramér's V

The chi-square statistic depends on sample size — a larger sample always gives a larger χ². **Cramér's V** measures the strength of association independent of sample size.

Compute Cramér's V to quantify the strength of association between gender and product preference.

In [ ]:
def cramers_v(contingency_table):
    """Compute Cramér's V for effect size of chi-square test."""
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    n = contingency_table.values.sum()
    min_dim = min(contingency_table.shape) - 1
    v = np.sqrt(chi2 / (n * min_dim))
    return v

v = cramers_v(observed)

print(f"Cramér's V = {v:.4f}")
print(f'\nInterpretation guide:')
print(f'  V ≈ 0.1: Small association')
print(f'  V ≈ 0.3: Medium association')
print(f'  V ≈ 0.5: Large association')
print(f'\n→ Our V = {v:.3f} indicates a {"small" if v < 0.2 else "medium" if v < 0.4 else "large"} association.')

---
## 9. Chi-Square Goodness of Fit

The **goodness of fit test** checks whether observed frequencies match expected frequencies from a theoretical distribution.

- H₀: The data follows the expected distribution
- H₁: The data does NOT follow the expected distribution

Python: `scipy.stats.chisquare(observed_freq, expected_freq)`

Test whether a die is fair: roll it 300 times and check if all faces appear with equal frequency.

In [ ]:
# Goodness of fit: Is this die fair?
np.random.seed(42)

# Simulate a slightly loaded die (face 6 appears more often)
loaded_probs = [0.14, 0.16, 0.16, 0.17, 0.17, 0.20]  # Slightly biased
rolls = np.random.choice([1, 2, 3, 4, 5, 6], size=300, p=loaded_probs)

# Observed frequencies
observed_freq = np.array([np.sum(rolls == i) for i in range(1, 7)])
expected_freq = np.array([300 / 6] * 6)  # Under H₀: fair die → 50 each

# Chi-square goodness of fit test
chi2_stat, p_value = stats.chisquare(observed_freq, expected_freq)

print('Die Roll Results (300 rolls):')
result_df = pd.DataFrame({
    'Face': range(1, 7),
    'Observed': observed_freq,
    'Expected': expected_freq,
    'O - E': observed_freq - expected_freq
})
print(result_df.to_string(index=False))

print(f'\nχ² = {chi2_stat:.4f}')
print(f'p-value = {p_value:.4f}')
print(f'Decision at α=0.05: {"Reject H₀" if p_value < 0.05 else "Fail to reject H₀"}')

Visualize the observed vs expected die roll frequencies.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(1, 7)
width = 0.35
ax.bar(x - width/2, observed_freq, width, color='steelblue', alpha=0.8, label='Observed')
ax.bar(x + width/2, expected_freq, width, color='coral', alpha=0.8, label='Expected (fair)')

ax.set_xlabel('Die Face', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title(f'Goodness of Fit Test: Is the Die Fair? (p = {p_value:.4f})', fontsize=13)
ax.set_xticks(x)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

Test whether customer arrival times follow a Poisson distribution — a more realistic goodness-of-fit application.

In [ ]:
# Goodness of fit: Do customer arrivals follow a Poisson distribution?
np.random.seed(42)

# Simulate hourly customer arrivals (actually Poisson with λ=5)
arrivals = np.random.poisson(lam=5, size=200)

# Observed frequency for each arrival count (0, 1, 2, ..., 10+)
bins = list(range(0, 11)) + [max(arrivals) + 1]  # Group 10+ together
observed_counts = np.histogram(arrivals, bins=bins)[0]

# Expected frequencies under Poisson(λ = sample mean)
lam_hat = arrivals.mean()  # MLE estimate of λ
expected_probs = [stats.poisson.pmf(k, lam_hat) for k in range(10)]
expected_probs.append(1 - sum(expected_probs))  # P(X >= 10)
expected_counts = np.array(expected_probs) * len(arrivals)

# Chi-square test (df = k - 1 - estimated_params = 11 - 1 - 1 = 9)
chi2_stat, p_value = stats.chisquare(observed_counts, expected_counts)

print(f'Sample mean λ̂ = {lam_hat:.2f}')
print(f'\n{"Arrivals":>10} {"Observed":>10} {"Expected":>10}')
print('-' * 32)
for i in range(11):
    label = f'{i}' if i < 10 else '10+'
    print(f'{label:>10} {observed_counts[i]:>10} {expected_counts[i]:>10.1f}')

print(f'\nχ² = {chi2_stat:.4f}, p = {p_value:.4f}')
print(f'→ The data {"does NOT" if p_value < 0.05 else "does"} deviate significantly from Poisson.')

---
## 10. Effect Size

A **statistically significant** result is not necessarily **practically important**. With a large enough sample, even a tiny, meaningless difference becomes "significant."

**Effect size** measures the *magnitude* of the difference, independent of sample size.

| Measure | Formula | Small | Medium | Large |
|---------|---------|-------|--------|-------|
| **Cohen's d** (means) | d = (x̄₁ - x̄₂) / s_pooled | 0.2 | 0.5 | 0.8 |
| **Pearson's r** (correlation) | r = correlation | 0.1 | 0.3 | 0.5 |
| **Cramér's V** (categorical) | V = √(χ²/n·min(r-1,c-1)) | 0.1 | 0.3 | 0.5 |
| **η² (eta-squared)** (ANOVA) | SS_between / SS_total | 0.01 | 0.06 | 0.14 |

### 10.1 Cohen's d

The most commonly used effect size measure for comparing two means.

Compute Cohen's d for the teaching methods comparison and interpret the effect size.

In [ ]:
def cohens_d(group1, group2):
    """Compute Cohen's d for two independent groups."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    
    # Pooled standard deviation
    s_pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    
    d = (group1.mean() - group2.mean()) / s_pooled
    return d

# Effect size for teaching methods
d = cohens_d(new_method, traditional)
t_stat, p_val = stats.ttest_ind(traditional, new_method, equal_var=False)

print(f'Teaching Methods Comparison:')
print(f'  Mean difference: {new_method.mean() - traditional.mean():.2f} points')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value: {p_val:.4f}')
print(f'  Cohen\'s d: {d:.4f}')
print(f'\n  Interpretation: {"Small" if abs(d) < 0.5 else "Medium" if abs(d) < 0.8 else "Large"} effect')

### 10.2 Statistical Significance vs Practical Significance

The classic demonstration: with a large enough sample, even a trivially small difference becomes "statistically significant."

Show that a meaningless 0.5-point difference becomes "significant" with n=10,000 per group, while a meaningful 5-point difference may not be significant with n=10.

In [ ]:
# Demonstration: significance ≠ importance
np.random.seed(42)

print('=== Large n, Tiny Effect ===')
# Huge sample, tiny difference (0.5 points)
large_a = np.random.normal(100, 15, 10_000)
large_b = np.random.normal(100.5, 15, 10_000)  # Only 0.5 difference!
t, p = stats.ttest_ind(large_a, large_b)
d = cohens_d(large_b, large_a)
print(f'n = 10,000 per group')
print(f'Mean diff = {large_b.mean() - large_a.mean():.2f}')
print(f'p-value = {p:.6f} → {"Significant!" if p < 0.05 else "Not significant"}')
print(f'Cohen\'s d = {d:.4f} → Trivially small effect!')

print(f'\n=== Small n, Large Effect ===')
# Small sample, big difference (5 points)
small_a = np.random.normal(100, 15, 10)
small_b = np.random.normal(105, 15, 10)  # 5-point difference
t, p = stats.ttest_ind(small_a, small_b)
d = cohens_d(small_b, small_a)
print(f'n = 10 per group')
print(f'Mean diff = {small_b.mean() - small_a.mean():.2f}')
print(f'p-value = {p:.4f} → {"Significant!" if p < 0.05 else "Not significant"}')
print(f'Cohen\'s d = {d:.4f} → {"Small" if abs(d) < 0.5 else "Medium" if abs(d) < 0.8 else "Large"} effect')

print(f'\n→ ALWAYS report effect size alongside p-values!')
print(f'→ Statistical significance alone is not enough.')

Visualize how p-values and effect sizes change with sample size to drive the point home.

In [ ]:
# How p-value shrinks with n for a FIXED tiny effect
np.random.seed(42)
true_diff = 1.0  # True tiny difference
sigma = 15
n_values = [10, 20, 50, 100, 200, 500, 1000, 2000, 5000]

p_values = []
d_values = []

for n in n_values:
    a = np.random.normal(100, sigma, n)
    b = np.random.normal(100 + true_diff, sigma, n)
    _, p = stats.ttest_ind(a, b)
    p_values.append(p)
    d_values.append(cohens_d(b, a))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# P-value vs n
axes[0].plot(n_values, p_values, 'bo-', markersize=8, linewidth=2)
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
axes[0].set_xlabel('Sample Size (n per group)', fontsize=12)
axes[0].set_ylabel('P-value', fontsize=12)
axes[0].set_title(f'P-value vs Sample Size\n(True diff = {true_diff}, σ = {sigma})', fontsize=13)
axes[0].set_xscale('log')
axes[0].legend(fontsize=11)

# Cohen's d stays constant regardless of n
axes[1].plot(n_values, d_values, 'go-', markersize=8, linewidth=2)
axes[1].axhline(true_diff/sigma, color='red', linestyle='--', linewidth=2, 
                label=f'True d = {true_diff/sigma:.3f}')
axes[1].set_xlabel('Sample Size (n per group)', fontsize=12)
axes[1].set_ylabel("Cohen's d", fontsize=12)
axes[1].set_title("Effect Size is Independent of Sample Size", fontsize=13)
axes[1].set_xscale('log')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## 11. Multiple Testing Problem

When we perform **many hypothesis tests** simultaneously, the chance of at least one false positive increases dramatically.

With m independent tests at α = 0.05:
- P(at least one false positive) = 1 - (1-α)^m
- For m = 20 tests: 1 - 0.95²⁰ = **64%** chance of at least one false positive!

### Correction Methods

| Method | Corrected α | Controls | Strictness |
|--------|-----------|----------|------------|
| **Bonferroni** | α/m | Family-wise error rate (FWER) | Very conservative |
| **Holm-Bonferroni** | Sequential α | FWER | Less conservative |
| **Benjamini-Hochberg** | Based on rank | False discovery rate (FDR) | Most liberal |

Demonstrate the multiple testing problem: run 20 tests on random data where H₀ is true for all, and watch false positives appear.

In [ ]:
# Multiple testing problem demonstration
np.random.seed(42)
n_tests = 20
n = 30  # Samples per test

# All null hypotheses are TRUE (both groups have same mean)
p_values = []
for i in range(n_tests):
    a = np.random.normal(100, 15, n)
    b = np.random.normal(100, 15, n)  # Same mean!
    _, p = stats.ttest_ind(a, b)
    p_values.append(p)

# How many false positives?
false_positives = sum(p < 0.05 for p in p_values)

print(f'Ran {n_tests} t-tests where ALL null hypotheses are TRUE')
print(f'False positives (p < 0.05): {false_positives}/{n_tests}')
print(f'\nP-values:')
for i, p in enumerate(p_values):
    marker = ' ← FALSE POSITIVE!' if p < 0.05 else ''
    print(f'  Test {i+1:2d}: p = {p:.4f}{marker}')

print(f'\nTheoretical: P(≥1 false positive in 20 tests) = {1 - 0.95**20:.1%}')

Apply Bonferroni and Benjamini-Hochberg corrections to the p-values and compare the results.

In [ ]:
from statsmodels.stats.multitest import multipletests

# Apply corrections
reject_bonf, pvals_bonf, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')
reject_holm, pvals_holm, _, _ = multipletests(p_values, alpha=0.05, method='holm')
reject_bh, pvals_bh, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

# Summary
results = pd.DataFrame({
    'Original p': p_values,
    'Bonferroni': pvals_bonf,
    'Holm': pvals_holm,
    'BH (FDR)': pvals_bh,
    'Reject (orig)': [p < 0.05 for p in p_values],
    'Reject (Bonf)': reject_bonf,
    'Reject (BH)': reject_bh
})

print('Multiple Testing Corrections:')
print(results[results['Reject (orig)']].to_string())

print(f'\nSummary:')
print(f'  Original:    {sum(p < 0.05 for p in p_values)} rejections (false positives!)')
print(f'  Bonferroni:  {sum(reject_bonf)} rejections')
print(f'  Holm:        {sum(reject_holm)} rejections')
print(f'  BH (FDR):    {sum(reject_bh)} rejections')
print(f'\n→ After correction, no false positives remain!')

Visualize the original vs corrected p-values to show how corrections raise the bar for significance.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(1, n_tests + 1)
width = 0.25

ax.bar(x - width, p_values, width, color='steelblue', alpha=0.8, label='Original')
ax.bar(x, pvals_bonf.clip(0, 1), width, color='coral', alpha=0.8, label='Bonferroni')
ax.bar(x + width, pvals_bh, width, color='seagreen', alpha=0.8, label='BH (FDR)')

ax.axhline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
ax.set_xlabel('Test Number', fontsize=12)
ax.set_ylabel('P-value', fontsize=12)
ax.set_title('Multiple Testing: Original vs Corrected P-values', fontsize=14)
ax.set_xticks(x)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

---
## 12. Practical Example: A/B Testing for a Website

An e-commerce company wants to test whether a redesigned checkout page (Version B) improves:
1. **Conversion rate** (proportion who complete purchase)
2. **Average order value** (among those who purchase)

We'll apply the full hypothesis testing workflow.

### Step 1: Generate A/B Test Data

Simulate realistic A/B test data with two versions of a checkout page.

Create a realistic A/B test dataset with user-level conversion and order value data.

In [ ]:
np.random.seed(42)

# Version A (control): 1500 visitors, 12% conversion, avg order $85
# Version B (redesign): 1500 visitors, 14.5% conversion, avg order $92
n_a, n_b = 1500, 1500

# Conversions (binary: 0 or 1)
conv_a = np.random.binomial(1, 0.12, n_a)
conv_b = np.random.binomial(1, 0.145, n_b)

# Order values (only for converters, lognormal distribution)
order_a = np.random.lognormal(np.log(85), 0.4, conv_a.sum())
order_b = np.random.lognormal(np.log(92), 0.4, conv_b.sum())

# Summary
print('=== A/B Test Data Summary ===')
print(f'\nVersion A (Control):')
print(f'  Visitors: {n_a}')
print(f'  Conversions: {conv_a.sum()} ({conv_a.mean():.1%})')
print(f'  Avg order value: ${order_a.mean():.2f}')

print(f'\nVersion B (Redesign):')
print(f'  Visitors: {n_b}')
print(f'  Conversions: {conv_b.sum()} ({conv_b.mean():.1%})')
print(f'  Avg order value: ${order_b.mean():.2f}')

### Step 2: Test Conversion Rate Difference

Use a two-proportion z-test (chi-square test on 2x2 table) to test if the conversion rate improved.

Perform a chi-square test on the 2x2 contingency table and a proportion z-test to check if Version B has a significantly higher conversion rate.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Method 1: Chi-square test on 2x2 table
contingency = pd.DataFrame(
    [[conv_a.sum(), n_a - conv_a.sum()],
     [conv_b.sum(), n_b - conv_b.sum()]],
    index=['Version A', 'Version B'],
    columns=['Converted', 'Not Converted']
)
print('Contingency Table:')
print(contingency)

chi2, p_chi2, dof, _ = stats.chi2_contingency(contingency)
print(f'\nChi-square test: χ² = {chi2:.4f}, p = {p_chi2:.4f}')

# Method 2: Two-proportion z-test (one-tailed: B > A)
count = np.array([conv_b.sum(), conv_a.sum()])
nobs = np.array([n_b, n_a])
z_stat, p_ztest = proportions_ztest(count, nobs, alternative='larger')

print(f'\nTwo-proportion z-test (H₁: p_B > p_A):')
print(f'  z = {z_stat:.4f}, p = {p_ztest:.4f}')
print(f'  Decision: {"Reject H₀" if p_ztest < 0.05 else "Fail to reject H₀"}')

### Step 3: Test Order Value Difference

Use a two-sample t-test and bootstrap to compare average order values between converters in A vs B.

Compare order values using both a Welch t-test and bootstrap CI, plus compute Cohen's d.

In [ ]:
# T-test on order values
t_stat, p_value = stats.ttest_ind(order_b, order_a, equal_var=False)
d = cohens_d(order_b, order_a)

print('=== Order Value Comparison ===')
print(f'Version A: mean = ${order_a.mean():.2f}, median = ${np.median(order_a):.2f}')
print(f'Version B: mean = ${order_b.mean():.2f}, median = ${np.median(order_b):.2f}')
print(f'\nWelch t-test: t = {t_stat:.4f}, p = {p_value:.4f}')
print(f'Cohen\'s d = {d:.4f} ({"Small" if abs(d) < 0.5 else "Medium" if abs(d) < 0.8 else "Large"} effect)')

# Bootstrap CI for the difference in means
n_boot = 10_000
boot_diffs = []
for _ in range(n_boot):
    ba = np.random.choice(order_a, size=len(order_a), replace=True)
    bb = np.random.choice(order_b, size=len(order_b), replace=True)
    boot_diffs.append(bb.mean() - ba.mean())

ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
print(f'\nBootstrap 95% CI for difference in means: (${ci_low:.2f}, ${ci_high:.2f})')
print(f'Does CI include 0? {ci_low <= 0 <= ci_high}')

### Step 4: Comprehensive Visualization

Create a dashboard summarizing all A/B test results.

Build a 4-panel figure showing conversion rates, order value distributions, bootstrap CIs, and a summary of key metrics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Conversion rates
ax = axes[0, 0]
rates = [conv_a.mean() * 100, conv_b.mean() * 100]
bars = ax.bar(['Version A\n(Control)', 'Version B\n(Redesign)'], rates, 
              color=['steelblue', 'coral'], alpha=0.8, width=0.5)
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
            f'{rate:.1f}%', ha='center', fontsize=14, fontweight='bold')
ax.set_ylabel('Conversion Rate (%)', fontsize=12)
ax.set_title(f'Conversion Rate (p = {p_ztest:.4f})', fontsize=13)
ax.set_ylim(0, max(rates) * 1.3)

# 2. Order value distributions
ax = axes[0, 1]
ax.hist(order_a, bins=30, alpha=0.6, color='steelblue', label='A', density=True, edgecolor='white')
ax.hist(order_b, bins=30, alpha=0.6, color='coral', label='B', density=True, edgecolor='white')
ax.axvline(order_a.mean(), color='steelblue', linestyle='--', linewidth=2)
ax.axvline(order_b.mean(), color='coral', linestyle='--', linewidth=2)
ax.set_xlabel('Order Value ($)', fontsize=12)
ax.set_title(f'Order Value Distribution (p = {p_value:.4f})', fontsize=13)
ax.legend(fontsize=11)

# 3. Bootstrap distribution of difference
ax = axes[1, 0]
ax.hist(boot_diffs, bins=50, color='seagreen', alpha=0.7, edgecolor='white', density=True)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No difference')
ax.axvline(ci_low, color='orange', linestyle=':', linewidth=2, label='95% CI')
ax.axvline(ci_high, color='orange', linestyle=':', linewidth=2)
ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 0.1], 
                 ci_low, ci_high, alpha=0.15, color='orange')
ax.set_xlabel('Difference in Mean Order Value (B - A)', fontsize=11)
ax.set_title('Bootstrap CI for Order Value Difference', fontsize=13)
ax.legend(fontsize=10)

# 4. Summary text
ax = axes[1, 1]
ax.axis('off')
summary_text = (
    f'A/B Test Summary Report\n'
    f'──────────────────────\n\n'
    f'Conversion Rate:\n'
    f'  A: {conv_a.mean():.1%}  →  B: {conv_b.mean():.1%}\n'
    f'  Lift: {(conv_b.mean()/conv_a.mean() - 1):.1%}\n'
    f'  p = {p_ztest:.4f} (one-tailed)\n\n'
    f'Avg Order Value:\n'
    f'  A: ${order_a.mean():.2f}  →  B: ${order_b.mean():.2f}\n'
    f'  Diff: ${order_b.mean() - order_a.mean():.2f}\n'
    f'  p = {p_value:.4f}, d = {d:.3f}\n'
    f'  95% CI: (${ci_low:.2f}, ${ci_high:.2f})\n\n'
    f'Recommendation:\n'
    f'  {"Deploy Version B" if p_ztest < 0.05 else "Need more data"}'
)
ax.text(0.1, 0.95, summary_text, transform=ax.transAxes, fontsize=12,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle('A/B Test Dashboard: Checkout Page Redesign', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Summary

| Section | Key Concepts | Python Functions |
|---------|-------------|------------------|
| 1. Framework | H₀/H₁, significance level α, decision rules | — |
| 2. P-values | P(data as extreme \| H₀ true), NOT P(H₀ true) | — |
| 3. Errors & Power | Type I (α), Type II (β), Power = 1-β | Manual computation |
| 4. Z-Test | σ known, z = (x̄-μ₀)/(σ/√n) | `stats.norm.cdf()` |
| 5. One-Sample T | σ unknown, most common | `stats.ttest_1samp(data, μ₀)` |
| 6. Two-Sample T | Compare two independent groups | `stats.ttest_ind(a, b)` |
| 7. Paired T | Before/after, matched pairs | `stats.ttest_rel(before, after)` |
| 8. Chi-Square Independence | Two categorical variables | `stats.chi2_contingency(table)` |
| 9. Chi-Square GoF | Observed vs expected frequencies | `stats.chisquare(obs, exp)` |
| 10. Effect Size | Cohen's d, Cramér's V | Manual formula |
| 11. Multiple Testing | Bonferroni, BH/FDR correction | `multipletests(pvals, method=)` |
| 12. Practical | A/B testing complete pipeline | All above combined |

### Key Takeaways

1. **Hypothesis testing is a framework for making decisions** — not for proving things
2. **p-value < 0.05 does not mean the result is important** — always report effect size
3. **"Fail to reject H₀" ≠ "H₀ is true"** — absence of evidence is not evidence of absence
4. **Use the right test**: paired data → paired test, categorical data → chi-square
5. **Correct for multiple testing** when running many tests simultaneously
6. **Always check assumptions** before interpreting results (normality, equal variances, independence)

### Decision Guide: Which Test to Use?

| Data Type | One Group | Two Groups (Independent) | Two Groups (Paired) |
|-----------|-----------|------------------------|--------------------|
| Continuous | One-sample t | Two-sample t (Welch) | Paired t |
| Categorical | Chi-square GoF | Chi-square independence | McNemar's test |

---
## 14. Homework

### Task 1: Drug Efficacy Test
A pharmaceutical company is testing a new blood pressure medication. They measured systolic BP before and after treatment for 30 patients:
```python
before_bp = [145, 152, 138, 160, 142, 155, 148, 137, 165, 150,
             143, 158, 141, 162, 147, 153, 139, 157, 144, 161,
             149, 156, 140, 163, 146, 154, 138, 159, 151, 142]
after_bp =  [138, 145, 135, 150, 140, 148, 142, 135, 155, 144,
             140, 150, 138, 153, 142, 147, 136, 150, 140, 152,
             143, 149, 137, 155, 141, 148, 135, 152, 145, 139]
```
1. State the null and alternative hypotheses (one-tailed: the drug reduces BP)
2. Perform a paired t-test
3. Compute the mean reduction and its 95% CI
4. Calculate Cohen's d for the paired differences
5. Visualize with a before/after plot and a histogram of differences

### Task 2: Customer Satisfaction by Region
A company surveyed customer satisfaction across three regions:
```python
satisfaction = pd.DataFrame({
    'Region': ['North']*40 + ['South']*35 + ['West']*45,
    'Satisfied': [1]*28 + [0]*12 + [1]*20 + [0]*15 + [1]*35 + [0]*10
})
```
1. Create a contingency table
2. Perform a chi-square test of independence
3. Compute Cramér's V
4. Which region has the highest/lowest satisfaction rate?
5. Visualize with a grouped bar chart

### Task 3: Multiple Comparisons
A food scientist tests whether 10 different spice extracts affect bacterial growth. For each extract, they compare a treatment group (n=15) to a control group (n=15).
1. Simulate this scenario where NONE of the extracts actually work (all effects are zero)
2. Run 10 independent two-sample t-tests
3. How many "significant" results do you find at α = 0.05?
4. Apply Bonferroni and Benjamini-Hochberg corrections
5. Repeat the simulation 1000 times and plot the distribution of false positive counts (before and after correction)

### Task 4: Complete A/B Test Analysis
An online learning platform tests two course page layouts:
- Layout A: 2000 visitors, 320 enrolled, average completion rate 45% (std 20%)
- Layout B: 2000 visitors, 380 enrolled, average completion rate 48% (std 22%)

1. Test whether Layout B has a higher enrollment rate (proportion test)
2. Among enrolled students, test whether Layout B has higher completion (t-test)
3. Compute effect sizes for both comparisons
4. Should the platform switch to Layout B? Justify with both statistical significance AND practical significance
5. How many more visitors would be needed to detect a 2% difference in enrollment with 80% power?

---
### Next Week Preview

**Week 9: Correlation & Regression** — We move from testing *differences* to modeling *relationships*. How strong is the association between study hours and exam scores? Can we predict sales from advertising spend? We'll cover Pearson and Spearman correlation, simple and multiple linear regression, residual diagnostics, and `statsmodels` OLS — the tools for understanding and predicting continuous outcomes.

---
*Applied Statistics with Python (2026) | Week 8 | Hypothesis Testing*